# Othello-GPT probing and intervention, ported to the transformer world model

**What this notebook does.** It reproduces the method of Li et al., *Emergent World
Representations: Exploring a Sequence Model Trained on a Synthetic Task* (ICLR 2023,
arXiv:2210.13382) — **the probe and the intervention exactly as they specify them** — on this
repo's causal transformer world model, and asks their question of our world: *is the world state
a probe reads causally responsible for what the model generates?*

Their pipeline has two halves and this notebook keeps both:

1. **§3 — probe.** Fit a classifier/regressor on the residual stream at each layer and compare a
   **linear** probe against a **2-layer MLP** (one hidden layer). Their headline is that the
   linear probe is poor and the MLP is good, so the world representation exists but is
   **nonlinear**.
2. **§4 — intervene.** Do gradient descent **on the activation** — `x' ← x − α ∂L(p_θ(x), B')/∂x`
   — so the probe reads a counterfactual state, applied at the **last timestep** and repeated at
   **every layer from a chosen starting layer `L_s` to the last**, alternating write and compute.
   Their headline is that this changes the model's output to match the counterfactual, i.e. the
   representation is **causal**.

Two probe targets are run, per Sevan's spec: one reading **positions only**, and one reading the
**entire world state at once** (positions *and* velocities for both objects), under the identical
edit objective — move one object's position, hold everything else where the probe already reads it.

**Why this is worth doing here.** Every editor in the editability thread so far is a *probe-derived
write* and every one fails. Othello-GPT is the strongest published claim that exactly this kind of
write *succeeds*. Running their method unchanged on our world separates "their method is better
than ours" from "their world is different from ours" — and those have very different consequences
for the thread's central negative.

**Sibling notebooks.** The transformer's two-state structure (carried buffer vs recomputed residual
stream) is established in [`../transformers/transformer_world_state.ipynb`](../transformers/transformer_world_state.ipynb).
Probe-gradient steering on the *input* surface is in
[`../input_grad_steering/`](../input_grad_steering/). This notebook is the *activation* surface with
the paper's exact schedule.

---

### What is copied exactly, and what necessarily differs

| Paper | Here | Why |
|---|---|---|
| 2-layer MLP probe `softmax(W₁ ReLU(W₂ x))`, vs linear `softmax(W x)` | same shapes, **linear read-out + squared error** | the board is 64 ternary tiles; our world state is continuous. Probe *shape* unchanged, loss and metric change to regression (R²) |
| probe accuracy reported per layer | same, per **residual point** | our model has `n_layers + 1 = 5` residual points |
| `x' ← x − α ∂L/∂x` on the activation | same rule | — |
| sequential write at layers `L_s … L`, last timestep, alternating write/compute | same | this is the paper's Figure 2C schedule; a single-layer write is explicitly *not* what they do |
| loss = target term + `β` × hold-the-rest term (App. G) | same, `β = 1.0` | this is what "move one object, keep the other fixed" means |
| null-intervention baseline (App. §4.2) | **Unsteered** arm | identical idea |
| split (activation, label) pairs 8:2 **at random** | split **by whole sequence** 80/20 | ⚠ **deliberate deviation.** Velocity is constant along a trajectory here, so a frame-level split leaves the identical label in train for every test frame — measured inflation **+0.34 R²** on velocity (`research/GOTCHAS.md`, 2026-08-14). Matching the paper would inflate every number in Fig 1 |
| plain gradient descent | **Adam** by default | the paper (App. G) states the process is "robust to different configurations of optimizer, learning rate α, and number of steps". Our residual points differ in scale ~17×, so no single raw-space α converges at every depth. §6 runs plain GD explicitly and reports where it differs |

## Definitions — every term and metric used below

Metric names, formulas and units are copied from the canonical registry
[`../METRICS_AND_EDITORS.md`](../METRICS_AND_EDITORS.md); they are computed by the shared module
`scripts/editability_metrics.py`, not re-derived here.

### Metrics

| name | formula | units | better | notes |
|---|---|---|---|---|
| **Edit Index** | per sample, over rays where the two ground-truth worlds differ: `(d_uned − d_edit) / (d_uned + d_edit)`, where `d_edit` = RMSE to the **edited** world and `d_uned` = RMSE to the **unedited** counterfactual | −1 … +1 | ↑ | **+1** = output matches the world where the teleport happened; **−1** = matches the world where it never did; **0** = equidistant, *which includes "the frame was destroyed"* — always read beside fidelity |
| **Edit Index by step** | the same quantity at each of the `K = 15` rollout steps | −1 … +1 | ↑ | landing an edit and holding it are different results; a step-0-only report can state the opposite of the truth |
| **Target RMSE** | RMSE(prediction, edited-world render) over rays the edited object **occupies after** the teleport | obs intensity | ↓ | did the object appear at the target? |
| **Ghost RMSE** | same, over rays it **vacated** | obs intensity | ↓ | did the object leave its old place? |
| **Collateral RMSE** | same, over rays the **other** object occupies | obs intensity | ↓ | was the object that should not move left alone? |
| **Edit-frame RMSE** | RMSE over **all** rays at step 0 | obs intensity | ↓ | overall fidelity of the edited frame |
| **GT-traj RMSE** | mean RMSE to the true post-edit observations across all `K` steps | obs intensity | ↓ | trajectory-level accuracy |
| **fidelity ratio** | `GT-traj RMSE(arm) / GT-traj RMSE(unsteered)` | ratio | ↓ | **> 1.05 flags an arm that scored by degrading the output rather than steering it.** Marked in figures, tabulated as a number |
| **probe R²** | `1 − ‖Y − p_θ(x)‖² / ‖Y − Ȳ_train‖²`, scored on **held-out sequences** against the **train** mean | — | ↑ | pooled over all output dims unless stated per-dim |
| **read-out error** | `√( Σ_dims w·(p_θ(x') − B')² )` after the intervention, `w` from the edit loss | sim units | ↓ | **did the optimisation land?** Separate question from whether the generation followed |
| **relative write size** | `‖x' − x‖ / ‖x‖` at the intervened residual point | ratio | — | how large a perturbation the write actually applied |

### Terms

| term | meaning |
|---|---|
| **residual point** `ℓ` | one of `n_layers + 1 = 5` places the residual stream can be read/written: `0` = encoder port `relu(Linear(obs))`, `1…3` = input to blocks 1–3, `4` = final pre-LayerNorm stream the decoder reads. An edit at point `ℓ` changes block inputs for layers **> ℓ only** |
| **applied layer** `L_s` | the paper's starting layer: the intervention is applied at `L_s` **and every residual point after it**, alternating write and compute |
| **edit frame** `ef = 20` | the frame at which one object teleports. The model is teacher-forced on `obs[0…ef−1]`, so **rollout step 0 decodes frame `ef`** |
| **`K = 15`** | post-edit rollout steps scored |
| **`β = 1.0`** | weight on the hold-the-rest term of the edit loss (paper App. G) |
| **late-t** | frames `t ≥ 15`, after the model's belief has converged; the repo's reporting convention for velocity |
| **Unsteered** | free-run from the same warmed state with no intervention — the paper's *null intervention* |
| **Oracle observation** | the model is teacher-forced one extra frame, the **real (noisy)** `obs[ef]`, so it simply *sees* the teleport. **Its rollout leads the others by one frame** — it is a reference for what any single-frame write could achieve, not a comparable arm |
| **position probe / full-state probe** | probe targets: 4 dims `(x, y)` per object, vs 8 dims `(x, y, vx, vy)` per object. **Both are edited with the identical objective** — move the teleported object's *position*, hold every other dim at the probe's own pre-intervention reading |

## Runs and arms used here

### World model — rows copied from [`../transformers/TRANSFORMER_RUNS.md`](../transformers/TRANSFORMER_RUNS.md)

No new world models were trained for this notebook.

| code | descriptive label (used in every figure) | window | carried `state_span` | params | best val |
|---|---|---|---|---|---|
| `W16` | **transformer · window 16** | 16 | 61 frames | 3,225,472 | 0.02359 |
| `W4` | **transformer · window 4** | 4 | 13 frames | 3,225,472 | 0.02372 |
| `W2` | **transformer · window 2** | 2 | 5 frames | 3,225,472 | 0.02396 |

`W16` is the primary run throughout: its `state_span` (61 frames) exceeds the 20 frames available
before the edit frame, so its effective carried state is the whole history — the most favourable
setting for a write to matter. §7 checks the headline against `W4` and `W2`.

Architecture (fixed): `d_model = 256` (**matched to the GRU's hidden size**), 4 layers, 4 heads,
RoPE, band-causal mask. Dataset `datasets/4_fixed_refl_inview`: 2 objects, 40 frames, `obs_res = 128`,
observation noise 0.2, position noise 0.04, edit frame 20, edits are in-frustum teleports of one object.

### Probe arms defined by this notebook

| arm | probe target | probe family | what it is |
|---|---|---|---|
| `linear · positions` | 4 dims | `Linear(256 → 4)`, closed-form least squares | paper §3.1 baseline |
| `MLP · positions` | 4 dims | `Linear(256 → 512) → ReLU → Linear(512 → 4)` | **paper §3.2 probe**, and the one that drives every edit |
| `linear · full state` | 8 dims | `Linear(256 → 8)` | positions **and** velocities together |
| `MLP · full state` | 8 dims | `Linear(256 → 512) → ReLU → Linear(512 → 8)` | the second edit driver |

All four are fit **once per residual point** (5 points), on 1,500 test sequences, held out by
sequence. The MLP is trained 200 epochs, Adam `lr = 1e-3`, on standardised inputs and targets.

> ⚠ **These are *not* the repo's standard readability probes.** `pim.extractors.fit_readability_probes`
> is a **2-hidden-layer × 256** network; the Othello paper's probe has **one** hidden layer. They are
> different objects and their R² values are not comparable — never quote one as the other
> (`harness/ANALYSIS.md` §1). The numbers here are labelled `MLP (512 hidden)` throughout to keep
> that distinction visible.

### Edit arms

| arm | what it is |
|---|---|
| **Unsteered** | free-run, no intervention — the paper's null-intervention baseline |
| **from `ℓ · …`** for `ℓ = 0…4` | the paper's intervention applied at residual point `ℓ` **and every point after it** |
| **Oracle observation** | fed the real post-edit frame; **leads by one frame**, reference only |

In [ ]:
# [1] Setup — imports, paths, provenance.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pim").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "notebooks/experiments/editability/othello_gpt"))
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

import othello_probe as op
import pipeline as pl
from pim.figures import waterfall_grid
from pim.figures.theme import PALETTE, style_ax

RUN = "W16"
N_SEQ = 1500  # test sequences for probe fitting (split by sequence, 80/20)
N_EDIT = 256  # held-out edit episodes scored
ALPHA = 0.05  # intervention step, RELATIVE to each residual point's activation scale
N_STEPS = 100  # gradient steps per residual point
BETA = 1.0  # weight on the hold-the-rest term (paper App. G)

bundle = pl.load(RUN)
N_POINTS = bundle.model.cfg.n_layers + 1
POINT_LABELS = [pl.residual_point_label(i) for i in range(N_POINTS)]

print(f"run             : {RUN}  (runs/transformers/{RUN}/best_model.pt)")
print(f"  epoch {bundle.info.epoch}, best val loss {bundle.info.val_loss:.5f}")
print(f"  d_model {bundle.model.cfg.d_model}, n_layers {bundle.model.cfg.n_layers}, "
      f"window {bundle.model.cfg.window}, state_span {bundle.model.state_span}")
print(f"  residual points : {N_POINTS} -> {POINT_LABELS}")
print(f"dataset         : datasets/4_fixed_refl_inview, edit frame {bundle.edits.edit_frame}, "
      f"K = {pl.K}, N_edit = {N_EDIT}")
print(f"device          : {pl.DEVICE}")
print(f"intervention    : alpha_rel {ALPHA} x act-scale, {N_STEPS} steps, beta {BETA}, Adam")

In [ ]:
# [2] Correctness gates. Everything downstream depends on these three, so they run first
#     and print numbers rather than being asserted silently.
model = bundle.model
ef = bundle.edits.edit_frame

# Gate 1 — the warmed state reproduces the one-pass teacher-forced prediction at frame ef.
# `state_from_obs` builds the carried state directly from the last `state_span` frames; if that
# disagreed with the training-time banded forward, every rollout below would start from the
# wrong place.
_o = torch.from_numpy(bundle.edits.obs[:64, : ef + 1]).float().to(pl.DEVICE)
_st = model.state_from_obs(_o[:, :ef])
with torch.no_grad():
    _d_state = model.decode(_st).cpu().numpy()
    _pred, _ = model(_o)
_gate1 = float(np.abs(_d_state - _pred[:, ef - 1].cpu().numpy()).max())

# Gate 2 — forcing residual point 4 to the value it already holds is a no-op, i.e. the
# intervention machinery is neutral when it writes nothing. This also confirms `Unsteered`
# (a plain free-run) is the right null baseline for the edited arms.
_setup_probe = pl.edit_setup(bundle, n_edit=64)
_free = pl.free_rollout(model, _setup_probe.state)
with torch.no_grad():
    _ident = model.rollout_with_edit(_setup_probe.state, 4, _setup_probe.x0[4], pl.K).cpu().numpy()
_gate2 = float(np.abs(_free - _ident).max())

# Gate 3 — the edits split contains exactly one intervention: the teleport under test.
_pos = bundle.edits.positions[:N_EDIT, :, : pl.N_OBJ, :].astype(np.float32)
_jump = np.linalg.norm(np.diff(_pos, axis=1), axis=-1).max(axis=-1)  # per-frame max object jump
_at_ef = _jump[:, ef - 1]
_elsewhere = np.delete(_jump, ef - 1, axis=1).max(axis=1)
_gate3 = float(_elsewhere.max())

print(f"Gate 1  state_from_obs.decode  vs one-pass forward : max|diff| = {_gate1:.2e}   (float tolerance)")
print(f"Gate 2  free-run vs identity activation write      : max|diff| = {_gate2:.2e}   (must be 0)")
print(f"Gate 3  largest object jump AT the edit frame      : {_at_ef.mean():.3f} (mean over episodes)")
print(f"        largest object jump at ANY OTHER frame     : {_gate3:.3f} (max over episodes)")
print(f"        -> one intervention per episode; nothing else moves discontinuously in the window")
assert _gate2 == 0.0, "identity write is not a no-op"

In [ ]:
# [3] Fit the paper's two probe families at every residual point, for both targets.
#     Held out BY SEQUENCE (80/20). ~25 s.
probes, stats = pl.probe_table(bundle, n_seq=N_SEQ, hidden=512, epochs=200, seed=0)


def stat(target, family, point):
    return next(
        s for s in stats if s["target"] == target and s["family"] == family and s["point"] == point
    )


FAMILIES = ["linear", "MLP (512 hidden)"]
print(f"fit {len(stats)} probes  ({len(FAMILIES)} families x {N_POINTS} residual points x 2 targets)")
print(f"train/test split: {int(0.8 * N_SEQ)}/{N_SEQ - int(0.8 * N_SEQ)} SEQUENCES\n")
for tname in ("pos", "full"):
    lab = "positions (4 dims)" if tname == "pos" else "full state (8 dims)"
    print(f"  {lab}")
    for fam in FAMILIES:
        r2 = [stat(tname, fam, i)["r2"] for i in range(N_POINTS)]
        print(f"    {fam:<18} R2 by point: " + "  ".join(f"{v:.3f}" for v in r2))

In [ ]:
# [4] Fig 1 — probe quality across residual points. The paper's Table 1 (linear) vs Table 2
#     (nonlinear) comparison, as a figure.
#     NOTE on panel (b): pooled R² is variance-weighted, and velocity has ~50x smaller variance
#     than position in sim units, so a pooled number for the full-state probe is almost entirely
#     the position dims. The equal-weight line and panel (c) are what actually show how well
#     velocity is read.
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.3), facecolor="white")
x = np.arange(N_POINTS)
STYLE = {"linear": dict(color=PALETTE[1], ls="--", marker="s"),
         "MLP (512 hidden)": dict(color=PALETTE[0], ls="-", marker="o")}

for ax, tname, title in [
    (axes[0], "pos", "(a) position probe — 4 dims"),
    (axes[1], "full", "(b) full-state probe — 8 dims"),
]:
    for fam in FAMILIES:
        ax.plot(x, [stat(tname, fam, i)["r2"] for i in range(N_POINTS)],
                label=f"{fam} — pooled", lw=2, ms=6, **STYLE[fam])
    if tname == "full":
        ax.plot(x, [np.mean(stat(tname, "MLP (512 hidden)", i)["per_dim_r2"])
                    for i in range(N_POINTS)],
                color=PALETTE[3], ls="-", marker="^", lw=2, ms=6,
                label="MLP — mean of per-dim R² (equal weight)")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("residual point (0 = encoder port, 4 = decoder input)")
    ax.set_ylabel("held-out R²  (by-sequence split, vs train mean)")
    ax.set_xticks(x)
    ax.set_xticklabels([lab.split(" · ")[0] for lab in POINT_LABELS])
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7.5, handlelength=2.6, loc="lower right")
    style_ax(ax)

best = int(np.argmax([stat("full", "MLP (512 hidden)", i)["r2"] for i in range(N_POINTS)]))
s_full = stat("full", "MLP (512 hidden)", best)
names = pl.TARGETS["full"]
ax = axes[2]
colors = [PALETTE[0]] * 4 + [PALETTE[3]] * 4
ax.barh(np.arange(8), s_full["per_dim_r2"], color=colors, edgecolor="#333", lw=0.5)
ax.set_yticks(np.arange(8))
ax.set_yticklabels(names, fontsize=9)
ax.invert_yaxis()
ax.axvline(0, color="#555", lw=1)
ax.set_xlim(-0.15, 1.0)
ax.set_xlabel("held-out R² per dimension")
ax.set_title(f"(c) full-state probe, per dimension\n(MLP, residual point {best})", fontsize=10)
ax.grid(alpha=0.25, axis="x")
handles = [plt.Rectangle((0, 0), 1, 1, color=PALETTE[0]),
           plt.Rectangle((0, 0), 1, 1, color=PALETTE[3])]
ax.legend(handles, ["position dims", "velocity dims"], fontsize=8, loc="lower right")
style_ax(ax)

fig.suptitle(
    "Fig 1 — how well each probe family reads the world state out of the residual stream, by depth\n"
    f"transformer · window 16 · {N_SEQ} test sequences · held out by sequence",
    fontsize=11.5, y=1.06,
)
fig.tight_layout()
plt.show()

_v = [s_full["per_dim_r2"][j] for j in range(4, 8)]
print(f"position dims R²: {min(s_full['per_dim_r2'][:4]):.3f} – {max(s_full['per_dim_r2'][:4]):.3f}")
print(f"velocity dims R²: {min(_v):.3f} – {max(_v):.3f}   <- pooled R² hides this entirely")

In [ ]:
# [5] Table 1 — probe quality, all families x residual points. Velocity is also reported
#     late-t (frames t >= 15), the repo convention, because the belief has not converged before then.
rows = [
    "| target | probe family | " + " | ".join(POINT_LABELS) + " |",
    "|---|---|" + "---|" * N_POINTS,
]
for tname, lab in (("pos", "positions"), ("full", "full state")):
    for fam in FAMILIES:
        cells = [f"{stat(tname, fam, i)['r2']:.3f}" for i in range(N_POINTS)]
        rows.append(f"| {lab} | {fam} | " + " | ".join(cells) + " |")
    if tname == "full":
        cells = [f"{stat(tname, 'MLP (512 hidden)', i)['r2_late']:.3f}" for i in range(N_POINTS)]
        rows.append("| full state *(late-t, t ≥ 15)* | MLP (512 hidden) | " + " | ".join(cells) + " |")
display(Markdown("**Table 1 — held-out R² by residual point**\n\n" + "\n".join(rows)))

pos_names = pl.TARGETS["full"]
r2d = stat("full", "MLP (512 hidden)", best)
r2l = stat("full", "MLP (512 hidden)", best)["per_dim_r2_late"]
rows2 = ["| dimension | all-t R² | late-t R² |", "|---|---|---|"]
for j, nm in enumerate(pos_names):
    rows2.append(f"| {nm} | {r2d['per_dim_r2'][j]:.3f} | {r2l[j]:.3f} |")
display(Markdown(
    f"**Table 2 — full-state probe, per dimension (MLP, residual point {best})**\n\n" + "\n".join(rows2)
))

_lin = max(stat("pos", "linear", i)["r2"] for i in range(N_POINTS))
_mlp = max(stat("pos", "MLP (512 hidden)", i)["r2"] for i in range(N_POINTS))
print(f"best position R²: linear {_lin:.3f}  ->  MLP {_mlp:.3f}   (gap {_mlp - _lin:+.3f})")
print("in-sample R² (overfit check, MLP position probe): "
      + "  ".join(f"{stat('pos', 'MLP (512 hidden)', i)['r2_insample']:.3f}" for i in range(N_POINTS)))

In [ ]:
# [6] Edit setup, then the step-size sweep. The step size is chosen by READ-OUT CONVERGENCE
#     (did the optimisation land the probe on the target?), never by Edit Index — picking it by
#     the outcome metric would select the setting that scores best by damaging the frame.
setup = pl.edit_setup(bundle, n_edit=N_EDIT)
print(f"edit episodes: {setup.n}   teleport distance: mean {setup.zones.teleport.mean():.3f} sim units")

ALPHA_GRID = [0.01, 0.02, 0.05, 0.1, 0.3, 1.0]
sweep = {}
for a in ALPHA_GRID:
    _, cards_a, recs_a = pl.run_arms(
        bundle, setup, probes, "pos", alpha=a, n_steps=N_STEPS, beta=BETA
    )
    UNSTEERED_EI = cards_a["Unsteered"]["edit_index"]  # identical for every alpha
    ORACLE_EI = cards_a["Oracle observation"]["edit_index"]
    sweep[a] = {
        ls: dict(
            readout=recs_a[ls][ls]["readout_err_after"],
            readout0=recs_a[ls][ls]["readout_err_before"],
            dx=recs_a[ls][ls]["delta_norm"] / recs_a[ls][ls]["x_norm"],
            ei=cards_a[f"from {pl.residual_point_label(ls)}"]["edit_index"],
            fid=cards_a[f"from {pl.residual_point_label(ls)}"]["fidelity_ratio"],
        )
        for ls in range(N_POINTS)
    }
print(f"swept {len(ALPHA_GRID)} step sizes x {N_POINTS} applied layers")
print(f"read-out error before any write : {sweep[ALPHA_GRID[0]][0]['readout0']:.3f} sim units")
print(f"unsteered Edit Index (step 0)   : {UNSTEERED_EI:+.3f}")
print(f"oracle-observation Edit Index   : {ORACLE_EI:+.3f}  (leads by one frame)")

In [ ]:
# [7] Fig 2 — the step-size sweep, and why the operating point is chosen on panel (a).
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.3), facecolor="white")
cols = [PALETTE[i] for i in range(N_POINTS)]

for ax, key, ylab, title in [
    (axes[0], "readout", "read-out error after the write (sim units)",
     "(a) did the optimisation land?"),
    (axes[1], "ei", "Edit Index at the edit frame (step 0)",
     "(b) did the generation follow?"),
    (axes[2], "dx", "relative write size  ‖x′ − x‖ / ‖x‖",
     "(c) how big was the write?"),
]:
    for ls in range(N_POINTS):
        ax.plot(ALPHA_GRID, [sweep[a][ls][key] for a in ALPHA_GRID],
                marker="o", ms=5, lw=1.8, color=cols[ls], label=POINT_LABELS[ls])
    ax.set_xscale("log")
    ax.set_xlabel("intervention step size α  (relative to activation scale)")
    ax.set_ylabel(ylab)
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.25)
    style_ax(ax)

axes[0].set_yscale("log")
axes[0].plot([], [], color="#555", ls=":", lw=1.5, label="before any write")
axes[0].axhline(sweep[ALPHA_GRID[0]][0]["readout0"], color="#555", ls=":", lw=1.5)

# Absolute Edit Index, with its own references marked on the SAME axis and in the same units.
axes[1].axhline(UNSTEERED_EI, color="#555", ls=":", lw=1.6)
axes[1].axhline(ORACLE_EI, color="#555", ls="-.", lw=1.6)
axes[1].plot([], [], color="#555", ls=":", lw=1.6, label="unsteered (no write)")
axes[1].plot([], [], color="#555", ls="-.", lw=1.6, label="oracle observation")
axes[1].set_ylim(-0.85, 0.35)

for ax in (axes[1], axes[2]):
    ax.axvline(ALPHA, color="#999", ls="--", lw=1.2)
axes[0].axvline(ALPHA, color="#999", ls="--", lw=1.2)
axes[0].plot([], [], color="#999", ls="--", lw=1.2, label="operating point used below")

axes[0].legend(fontsize=7, handlelength=2.4, ncol=2, loc="upper left")
axes[1].legend(fontsize=7, handlelength=2.4, ncol=2, loc="upper left")

fig.suptitle(
    "Fig 2 — intervention step size: read-out convergence, generation response, and write magnitude\n"
    "position probe · one line per applied layer L_s · dashed vertical = the setting used below",
    fontsize=11.5, y=1.05,
)
fig.tight_layout()
plt.show()

_best_a = min(ALPHA_GRID, key=lambda a: np.mean([sweep[a][ls]["readout"] for ls in range(N_POINTS)]))
print(f"read-out error is minimised at α = {_best_a} and RISES beyond it (the write overshoots the")
print("target), while the Edit Index keeps climbing — so any gain past that point is bought with")
print("write magnitude, not with a more accurate read-out. The operating point is therefore chosen")
print(f"on panel (a), where the optimisation actually converges: α = {ALPHA}.")

In [ ]:
# [8] Run every arm at the chosen operating point, for both probe targets.
ROLLS, CARDS, RECS = {}, {}, {}
for tname in ("pos", "full"):
    ROLLS[tname], CARDS[tname], RECS[tname] = pl.run_arms(
        bundle, setup, probes, tname, alpha=ALPHA, n_steps=N_STEPS, beta=BETA
    )

ARMS = list(ROLLS["pos"].keys())
print("arms:", ARMS)
for tname in ("pos", "full"):
    lab = "position probe" if tname == "pos" else "full-state probe"
    print(f"\n{lab}")
    for k in ARMS:
        c = CARDS[tname][k]
        rd = ""
        if k.startswith("from "):
            ls = int(k.split()[1])
            rd = f"  read-out {RECS[tname][ls][ls]['readout_err_after']:.3f}"
        print(f"  {k:<30} EI(step 0) {c['edit_index']:+.3f}   EI(step 14) "
              f"{c['edit_index_by_step'][-1]:+.3f}   fidelity {c['fidelity_ratio']:.3f}{rd}")

In [ ]:
# [9] Fig 3 — waterfall, position probe. Built through the one canonical helper
#     (pim.figures.waterfall_grid) so the spec cannot drift: gray on dark, noisy context frames
#     above the edit line, EACH column its own free-run from step 0, fixed vmin/vmax, GT first.
#
#     SAMPLES ARE DRAWN AT RANDOM (seeded). An earlier version of this notebook used the four
#     LARGEST teleports, which is a biased draw: this editor's effect grows with teleport size
#     while the unsteered baseline is flat, so those episodes sat at the 98th percentile of the
#     Edit Index distribution (+0.07 against the -0.54 mean) and the panel read as typical when
#     it was the best case. Fig 3b below shows that selection, labelled as such.
SAMPLES = pl.random_samples(setup.n, k=4, seed=0)
SAMPLES_LARGEST = [int(i) for i in np.argsort(setup.zones.teleport)[::-1][:4]]


def _per_sample_index(pred0, z):
    out = np.full(len(pred0), np.nan)
    for i in range(len(pred0)):
        m = z.differing[i]
        if not m.any():
            continue
        de = np.sqrt(((pred0[i, m] - z.gt_edited[i, m]) ** 2).mean())
        du = np.sqrt(((pred0[i, m] - z.gt_unedited[i, m]) ** 2).mean())
        out[i] = (du - de) / (du + de)
    return out


EI_PER_SAMPLE = _per_sample_index(
    ROLLS["pos"]["from " + pl.residual_point_label(0)][:, 0], setup.zones)


def show_waterfall(tname, fignum, samples, selection_note):
    lab = "position probe (4 dims)" if tname == "pos" else "full-state probe (8 dims)"
    fig = waterfall_grid(
        {k: ROLLS[tname][k] for k in ARMS},
        setup.ctx,
        setup.gt_roll,
        title=(f"Fig {fignum} - what each intervention actually generates - {lab}\n"
               f"{selection_note}\n"
               f"transformer - window 16 - N = {setup.n} edits - alpha = {ALPHA} - "
               f"column titles are the MEAN over all {setup.n} episodes"),
        sample_idx=samples,
        target_x=setup.tgt_cx,
        ghost_x=setup.ghost_cx,
        metrics={k: CARDS[tname][k]["edit_index"] for k in ARMS},
        metric_label="Edit Index (mean, all N)",
        leads_by_one=("Oracle observation",),
        gt_label="GT (sim clean obs)",
    )
    plt.show()
    print(f"   samples {samples} - teleport "
          f"{[f'{setup.zones.teleport[i]:.2f}' for i in samples]} "
          f"(population range {setup.zones.teleport.min():.2f}-"
          f"{setup.zones.teleport.max():.2f}, mean {setup.zones.teleport.mean():.2f})")
    print(f"   per-sample Edit Index of the 'from point 0' arm on these rows: "
          f"{[f'{EI_PER_SAMPLE[i]:+.2f}' for i in samples]}   "
          f"population mean {np.nanmean(EI_PER_SAMPLE):+.3f}, "
          f"median {np.nanmedian(EI_PER_SAMPLE):+.3f}")


show_waterfall("pos", 3, SAMPLES,
               "RANDOM sample of 4 episodes (seed 0) - representative, not selected")

In [ ]:
# [9b] Fig 3b — THE SAME ARMS ON THE FOUR LARGEST TELEPORTS. This is a deliberately
#      BIASED selection, shown only so the best case is visible next to the typical one.
#      Do not read it as representative: these four episodes sit at the 98th percentile.
show_waterfall("pos", "3b", SAMPLES_LARGEST,
               "LARGEST-TELEPORT episodes - BEST CASE, NOT TYPICAL (98th percentile; "
               "population mean Edit Index is far lower)")
print()
print("   why this selection flatters the editor - Edit Index by teleport quartile:")
q = np.argsort(setup.zones.teleport)
for j, g in enumerate(np.array_split(q, 4)):
    ei_e = np.nanmean(EI_PER_SAMPLE[g])
    ei_u = np.nanmean(_per_sample_index(ROLLS["pos"]["Unsteered"][:, 0], setup.zones)[g])
    print(f"     quartile {j + 1} (mean teleport {setup.zones.teleport[g].mean():.2f}): "
          f"edited {ei_e:+.3f}   unsteered {ei_u:+.3f}")
print("   the edited arm improves with teleport size; the unsteered baseline does not.")

In [ ]:
# [10] Fig 4 — waterfall, full-state probe. Same arms, same RANDOM samples, same spec: the only
#      change is that the probe driving the write also reads velocity.
show_waterfall("full", 4, SAMPLES,
               "RANDOM sample of 4 episodes (seed 0) - the same episodes as Fig 3")

In [ ]:
# [11] Fig 5 — Edit Index by applied layer, absolute (not a gain), with each arm's own
#      references marked on the same axis. Both panels share one category order so they can be
#      scanned horizontally.
EDIT_ARMS = [a for a in ARMS if a.startswith("from ")]
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4), facecolor="white", sharex=True)
y = np.arange(len(EDIT_ARMS))

for ax, tname, title in [
    (axes[0], "pos", "(a) position probe — 4 dims"),
    (axes[1], "full", "(b) full-state probe — 8 dims"),
]:
    vals = [CARDS[tname][a]["edit_index"] for a in EDIT_ARMS]
    fids = [CARDS[tname][a]["fidelity_ratio"] for a in EDIT_ARMS]
    bars = ax.barh(y, vals, color=[PALETTE[i] for i in range(len(EDIT_ARMS))],
                   edgecolor="#333", lw=0.6)
    # the fidelity guard: mark only the arms that FAIL it, one cue, one legend entry
    for i, (b, f) in enumerate(zip(bars, fids)):
        if f > 1.05:
            ax.plot(vals[i], i, marker="x", ms=11, mew=2.5, color="#d55e00", zorder=5)
    ax.axvline(UNSTEERED_EI, color="#555", ls=":", lw=1.8)
    ax.axvline(ORACLE_EI, color="#555", ls="-.", lw=1.8)
    ax.axvline(0.0, color="#999", lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels(EDIT_ARMS, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(-0.85, 0.35)
    ax.set_xlabel("Edit Index at the edit frame (step 0)   −1 … +1, ↑ better")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.25, axis="x")
    style_ax(ax)

h = [plt.Line2D([], [], color="#555", ls=":", lw=1.8),
     plt.Line2D([], [], color="#555", ls="-.", lw=1.8),
     plt.Line2D([], [], color="#d55e00", marker="x", ls="none", ms=10, mew=2.5)]
fig.legend(h, ["unsteered (no write)", "oracle observation (leads by one frame)",
               "fidelity ratio > 1.05 (scored by degrading)"],
           loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False,
           fontsize=8.5, handlelength=2.6)
fig.suptitle(
    "Fig 5 — Edit Index by the layer the intervention starts from\n"
    f"applied at that residual point and every point after it · α = {ALPHA} · N = {setup.n} edits",
    fontsize=11.5, y=1.14,
)
fig.tight_layout()
plt.show()

In [ ]:
# [12] Fig 6 — Edit Index ACROSS the rollout, not only at step 0. Landing an edit and holding it
#      are different results. Note the unsteered curve climbs on its own as a free-run drifts away
#      from BOTH reference worlds, so it is drawn on the same axes as the arm it is the
#      reference for — the gap, not the raw value, is what "distinguishable from doing nothing" means.
steps = np.arange(pl.K)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4), facecolor="white", sharey=True)

for ax, tname, title in [
    (axes[0], "pos", "(a) position probe — 4 dims"),
    (axes[1], "full", "(b) full-state probe — 8 dims"),
]:
    for i, a in enumerate(EDIT_ARMS):
        ax.plot(steps, CARDS[tname][a]["edit_index_by_step"], lw=2, marker="o", ms=3.5,
                color=PALETTE[i], label=a)
    ax.plot(steps, CARDS[tname]["Unsteered"]["edit_index_by_step"],
            color="#555", ls=":", lw=2.2, label="unsteered (no write)")
    ax.plot(steps, CARDS[tname]["Oracle observation"]["edit_index_by_step"],
            color="#555", ls="-.", lw=2.2, label="oracle observation (leads by one frame)")
    ax.axhline(0.0, color="#999", lw=1)
    ax.set_xlabel("rollout step   (step 0 = the edit frame)")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.25)
    style_ax(ax)

axes[0].set_ylabel("Edit Index   −1 … +1, ↑ better")
axes[0].set_ylim(-0.8, 0.3)
hs, ls_ = axes[0].get_legend_handles_labels()
fig.legend(hs, ls_, loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=4, frameon=False,
           fontsize=8, handlelength=2.6)
fig.suptitle(
    "Fig 6 — does the edit hold? Edit Index at every rollout step\n"
    f"transformer · window 16 · N = {setup.n} edits · α = {ALPHA}",
    fontsize=11.5, y=1.16,
)
fig.tight_layout()
plt.show()

for tname in ("pos", "full"):
    b = CARDS[tname]["from " + pl.residual_point_label(0)]["edit_index_by_step"]
    u = CARDS[tname]["Unsteered"]["edit_index_by_step"]
    print(f"{tname:>5}: best arm step 0 {b[0]:+.3f} -> step 14 {b[-1]:+.3f} | "
          f"unsteered {u[0]:+.3f} -> {u[-1]:+.3f} | gap {b[0]-u[0]:+.3f} -> {b[-1]-u[-1]:+.3f}")

In [ ]:
# [13] Table 3 — the full canonical scorecard for every arm, both targets.
#      Now carries DIRECTION COSINE and ACHIEVED FRACTION beside the Edit Index. The index is a
#      ratio of distances and cannot tell "the change pointed the wrong way" from "the change
#      pointed the right way and was a few percent of the required size" — and here it is the
#      second. Added 2026-08-18; see METRICS_AND_EDITORS.md.
from editability_metrics import direction_report  # noqa: E402

DIRS = {t_: {a: direction_report(ROLLS[t_][a][:, 0], ROLLS[t_]["Unsteered"][:, 0], setup.zones)
             for a in ARMS} for t_ in ("pos", "full")}
for tname in ("pos", "full"):
    lab = "position probe (4 dims)" if tname == "pos" else "full-state probe (8 dims)"
    rows = [
        "| arm | Edit Index (step 0) | Edit Index (step 14) | Target RMSE ↓ | Ghost RMSE ↓ "
        "| Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ratio | direction cos (angle) "
        "| achieved fraction | read-out error ↓ |",
        "|---|---|---|---|---|---|---|---|---|---|",
    ]
    for a in ARMS:
        c = CARDS[tname][a]
        rd = "—"
        if a.startswith("from "):
            ls = int(a.split()[1])
            rd = f"{RECS[tname][ls][ls]['readout_err_after']:.3f}"
        note = " ⚠" if c["fidelity_ratio"] > 1.05 else ""
        d = DIRS[tname][a]
        dr = "—" if a == "Unsteered" else f"{d['direction_cos']:+.3f} ({d['direction_angle_deg']:.0f}°)"
        af = "—" if a == "Unsteered" else f"{d['achieved_fraction']:.3f}"
        rows.append(
            f"| {a} | {c['edit_index']:+.3f} | {c['edit_index_by_step'][-1]:+.3f} "
            f"| {c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} "
            f"| {c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.3f}{note} "
            f"| {dr} | {af} | {rd} |"
        )
    display(Markdown(f"**Table 3 — canonical §4 scorecard, {lab}**\n\n" + "\n".join(rows)))

print("read-out error is in sim units and starts at "
      f"{RECS['pos'][0][0]['readout_err_before']:.2f} before any write.")
print("Target RMSE for the oracle observation is the reference for 'the object actually appeared'.")
print(f"direction-cosine chance level (shuffled pairs): "
      f"{DIRS['pos']['from ' + pl.residual_point_label(0)]['direction_cos_shuffled']:+.3f}")
print("-> the write's change is DIRECTIONALLY well above chance while being a few percent of the")
print("   required magnitude. The Edit Index reports only the magnitude half of that.")

In [ ]:
# [14] Fig 7 — plain gradient descent, the paper's literal rule, alongside Adam. The paper states
#      the process is robust to the optimiser; this checks that on our model. GD needs a step size
#      per residual point spanning three orders of magnitude, so it is swept per point and the
#      setting with the LOWEST read-out error is kept — the same selection rule used for Adam.
GD_GRID = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0]
gd = {}
for ls in range(N_POINTS):
    bestrec = None
    for a in GD_GRID:
        rec = {}
        r = pl.op.rollout_with_sequential_intervention(
            model, setup.state, {i: probes[("pos", i)] for i in range(N_POINTS)},
            {i: pl.op.build_edit_spec(probes[("pos", i)], setup.x0[i],
                                      setup.change_mask["pos"], setup.target_values["pos"], beta=BETA)
             for i in range(N_POINTS)},
            ls, pl.K, alpha=a, n_steps=N_STEPS, optimizer="gd", record=rec,
        ).cpu().numpy()
        rd = rec[ls]["readout_err_after"]
        if not np.isfinite(rd):
            continue
        c = pl.edit_scorecard(r, setup.zones, setup.gt_roll)
        cand = dict(alpha=a, readout=rd, ei=c["edit_index"],
                    dx=rec[ls]["delta_norm"] / rec[ls]["x_norm"],
                    fid=pl.fidelity_ratio(c, CARDS["pos"]["Unsteered"]))
        if bestrec is None or rd < bestrec["readout"]:
            bestrec = cand
    gd[ls] = bestrec

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3), facecolor="white")
w = 0.38
xs = np.arange(N_POINTS)
for ax, key, ylab, title in [
    (axes[0], "readout", "read-out error after the write (sim units)",
     "(a) read-out error after the write"),
    (axes[1], "dx", "relative write size  ‖x′ − x‖ / ‖x‖",
     "(b) size of the write that was applied"),
]:
    ax.bar(xs - w / 2, [sweep[ALPHA][ls][key] for ls in range(N_POINTS)], w,
           label="Adam (used throughout)", color=PALETTE[0], edgecolor="#333", lw=0.6)
    ax.bar(xs + w / 2, [gd[ls][key] for ls in range(N_POINTS)], w,
           label="plain gradient descent (paper's literal rule)", color=PALETTE[2],
           edgecolor="#333", lw=0.6)
    ax.set_xticks(xs)
    ax.set_xticklabels([lab.split(" · ")[0] for lab in POINT_LABELS])
    ax.set_xlabel("applied layer L_s (residual point)")
    ax.set_ylabel(ylab)
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.25, axis="y")
    ax.legend(fontsize=8, handlelength=2.2)
    style_ax(ax)
axes[0].set_yscale("log")

fig.suptitle(
    "Fig 7 — optimiser check: the paper's plain gradient descent vs Adam, at matched selection rule\n"
    "step size chosen per residual point by lowest read-out error in both cases · position probe",
    fontsize=11.5, y=1.07,
)
fig.tight_layout()
plt.show()

rows = ["| applied layer | GD step size | GD read-out | GD Edit Index | GD write size "
        "| Adam read-out | Adam Edit Index | Adam write size |", "|---|---|---|---|---|---|---|---|"]
for ls in range(N_POINTS):
    g, ad = gd[ls], sweep[ALPHA][ls]
    rows.append(f"| {POINT_LABELS[ls]} | {g['alpha']:g} | {g['readout']:.3f} | {g['ei']:+.3f} "
                f"| {g['dx']:.3f} | {ad['readout']:.3f} | {ad['ei']:+.3f} | {ad['dx']:.3f} |")
display(Markdown("**Table 4 — optimiser comparison, position probe**\n\n" + "\n".join(rows)))

In [ ]:
# [15] Fig 8 — does the headline depend on how much history the model carries? `window` dials
#      between "history must be compressed into the residual stream" (W2) and "effectively a
#      lookup over raw history" (W16). Full pipeline re-run per window. ~1 min.
WINDOWS = ["W2", "W4", "W16"]
WIN_LABEL = {w: f"transformer · window {w[1:]}" for w in WINDOWS}
win = {}
for w in WINDOWS:
    bw = pl.load(w)
    pw, sw = pl.probe_table(bw, n_seq=N_SEQ, hidden=512, epochs=200, seed=0)
    su = pl.edit_setup(bw, n_edit=N_EDIT)
    _, cw, rw = pl.run_arms(bw, su, pw, "pos", alpha=ALPHA, n_steps=N_STEPS, beta=BETA)
    win[w] = dict(
        cards=cw,
        probe_r2=[next(s["r2"] for s in sw if s["target"] == "pos"
                       and s["family"] == "MLP (512 hidden)" and s["point"] == i)
                  for i in range(N_POINTS)],
        state_span=bw.model.state_span,
    )
    print(f"{w}: state_span {bw.model.state_span:>3}  best probe R² "
          f"{max(win[w]['probe_r2']):.3f}  unsteered {cw['Unsteered']['edit_index']:+.3f}  "
          f"best arm {max(cw[a]['edit_index'] for a in EDIT_ARMS):+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3), facecolor="white")
for i, w in enumerate(WINDOWS):
    axes[0].plot(np.arange(N_POINTS), win[w]["probe_r2"], marker="o", lw=2, ms=5,
                 color=PALETTE[i], label=WIN_LABEL[w])
    axes[1].plot(np.arange(N_POINTS), [win[w]["cards"][a]["edit_index"] for a in EDIT_ARMS],
                 marker="o", lw=2, ms=5, color=PALETTE[i], label=WIN_LABEL[w])
    axes[1].axhline(win[w]["cards"]["Unsteered"]["edit_index"], color=PALETTE[i], ls=":", lw=1.4)
axes[1].plot([], [], color="#555", ls=":", lw=1.4, label="that model's own unsteered index")
for ax, ylab, title, ylim in [
    (axes[0], "held-out R² (MLP position probe)", "(a) how readable is position?", (0.0, 1.0)),
    (axes[1], "Edit Index at the edit frame (step 0)", "(b) how much does the write move it?", (-0.85, 0.1)),
]:
    ax.set_xticks(np.arange(N_POINTS))
    ax.set_xticklabels([lab.split(" · ")[0] for lab in POINT_LABELS])
    ax.set_xlabel("residual point  (panel b: applied layer L_s)")
    ax.set_ylabel(ylab)
    ax.set_title(title, fontsize=10)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, handlelength=2.6, loc="lower right")
    style_ax(ax)
fig.suptitle(
    "Fig 8 — the same experiment across attention windows (how much history the model carries)\n"
    f"position probe · α = {ALPHA} · N = {N_EDIT} edits · dotted = each model's own unsteered index",
    fontsize=11.5, y=1.07,
)
fig.tight_layout()
plt.show()

In [ ]:
# [16] Edit Index audit. The edit frame *looks* close to the ground truth in Fig 3, yet the index
#      is strongly negative. This checks whether that is a defect in the metric or a real property
#      of the output: (a) recompute the index from scratch with no shared code, (b) report how much
#      of the frame the index actually looks at, (c) compare full-frame error against error
#      restricted to the rays that distinguish the two worlds.
z = setup.zones
uns0 = ROLLS["pos"]["Unsteered"][:, 0]
edt0 = ROLLS["pos"]["from " + pl.residual_point_label(0)][:, 0]
allm = np.ones_like(z.differing)


def rms(a, b_, m):
    return float(np.sqrt(((a - b_) ** 2)[m].mean()))


def ei_from_scratch(pred, ge, gu, mask):
    """Independent re-implementation — deliberately does not import the shared module."""
    vals = []
    for i in range(len(pred)):
        m = mask[i]
        if not m.any():
            continue
        de = np.sqrt(((pred[i, m] - ge[i, m]) ** 2).mean())
        du = np.sqrt(((pred[i, m] - gu[i, m]) ** 2).mean())
        vals.append((du - de) / (du + de))
    return float(np.mean(vals)), len(vals)


print("(a) independent recomputation of the Edit Index")
for nm, arr in [("Unsteered", uns0), ("from " + pl.residual_point_label(0), edt0)]:
    v, nscored = ei_from_scratch(arr, z.gt_edited, z.gt_unedited, z.differing)
    print(f"    {nm:<24} scorecard {CARDS['pos'][nm]['edit_index']:+.4f}   "
          f"from scratch {v:+.4f}   (scored on {nscored}/{setup.n} episodes)")

n_empty = int((~z.differing.any(1)).sum())
med = int(np.median(z.differing.sum(1)))
print("\n(b) how much of the frame the index looks at")
print(f"    rays where the two ground-truth worlds differ: median {med} of "
      f"{z.differing.shape[1]}  ({med / z.differing.shape[1]:.0%} of the frame)")
print(f"    episodes with an EMPTY differing mask (teleport invisible in observation space): "
      f"{n_empty} -> silently dropped from the mean")

print("\n(c) full frame vs the differing rays only  (RMSE, obs intensity)")
hdr = f"    {'':<24}{'':<12}{'vs EDITED world':>16}{'vs UNEDITED world':>19}"
print(hdr)
for nm, arr in [("unsteered", uns0), ("edited (from point 0)", edt0)]:
    print(f"    {nm:<24}{'full frame':<12}{rms(arr, z.gt_edited, allm):>16.4f}"
          f"{rms(arr, z.gt_unedited, allm):>19.4f}")
    print(f"    {'':<24}{'differing':<12}{rms(arr, z.gt_edited, z.differing):>16.4f}"
          f"{rms(arr, z.gt_unedited, z.differing):>19.4f}")
print(f"    {'the two GT worlds':<24}{'full frame':<12}"
      f"{rms(z.gt_edited, z.gt_unedited, allm):>16.4f}")
print(f"    {'':<24}{'differing':<12}{rms(z.gt_edited, z.gt_unedited, z.differing):>16.4f}")
print("\n    -> the frame looks close to the edited world overall because the two worlds SHARE")
print("       ~78% of their rays (the untouched object plus background). The index deliberately")
print("       excludes those, and on the rays that carry the edit the unsteered output is ~6x")
print("       closer to the UNEDITED world. Both readings are correct; they measure different things.")

In [ ]:
# [17] Fig 9 — the edit frame on its own, as intensity curves rather than a waterfall row.
#      Four references on one axis: the two ground-truth worlds (the edit is the difference
#      between them), and the model's output with and without the intervention. The shaded band
#      is the ONLY region the Edit Index scores.
fig, axes = plt.subplots(2, len(SAMPLES), figsize=(4.4 * len(SAMPLES), 7.4), facecolor="white",
                         sharex=True)
rays = np.arange(z.gt_edited.shape[1])

for c, smp in enumerate(SAMPLES):
    m = z.differing[smp]
    for r, (arm_key, arm_lab) in enumerate([
        ("Unsteered", "model, no intervention"),
        ("from " + pl.residual_point_label(0), "model, intervention from point 0"),
    ]):
        ax = axes[r][c]
        # the scored support, drawn first so the curves sit on top
        ax.fill_between(rays, 0, 1, where=m, color="#cfd8e8", alpha=0.75, step="mid",
                        label="rays the Edit Index scores", zorder=0)
        ax.plot(rays, z.gt_unedited[smp], color="#555", ls=":", lw=2.0,
                label="GT — unedited world (no teleport)")
        ax.plot(rays, z.gt_edited[smp], color="#111", ls="-", lw=2.0,
                label="GT — edited world (the target)")
        ax.plot(rays, ROLLS["pos"][arm_key][smp, 0], color=PALETTE[1 if r else 0], lw=1.8,
                label=arm_lab)
        ax.axvline(setup.tgt_cx[smp], color="#00b050", lw=1.4)
        ax.axvline(setup.ghost_cx[smp], color="#d55e00", ls="--", lw=1.4)

        de = np.sqrt(((ROLLS["pos"][arm_key][smp, 0][m] - z.gt_edited[smp][m]) ** 2).mean())
        du = np.sqrt(((ROLLS["pos"][arm_key][smp, 0][m] - z.gt_unedited[smp][m]) ** 2).mean())
        ax.set_title(f"sample {smp} · {arm_lab}\n"
                     f"d(edited) {de:.3f} · d(unedited) {du:.3f} · "
                     f"Edit Index {(du - de) / (du + de):+.3f}", fontsize=8.5)
        ax.set_ylim(-0.02, 1.02)
        ax.grid(alpha=0.25)
        style_ax(ax)
        if c == 0:
            ax.set_ylabel("observation intensity")
        if r == 1:
            ax.set_xlabel("ray index (the 1D scan)")

h = [plt.Line2D([], [], color="#555", ls=":", lw=2.0),
     plt.Line2D([], [], color="#111", ls="-", lw=2.0),
     plt.Line2D([], [], color=PALETTE[0], lw=1.8),
     plt.Line2D([], [], color=PALETTE[1], lw=1.8),
     plt.Rectangle((0, 0), 1, 1, color="#cfd8e8"),
     plt.Line2D([], [], color="#00b050", lw=1.4),
     plt.Line2D([], [], color="#d55e00", ls="--", lw=1.4)]
lab = ["GT — unedited world", "GT — edited world (target)", "model, no intervention",
       "model, intervention from point 0", "rays the Edit Index scores", "target", "ghost"]
fig.legend(h, lab, loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=4, frameon=False,
           fontsize=8.5, handlelength=2.6)
fig.suptitle(
    "Fig 9 — the edit frame alone: both ground-truth worlds against the model with and without the write\n"
    "RANDOM sample of 4 episodes (seed 0) — representative, not selected\n"
    f"transformer · window 16 · rollout step 0 (frame {ef}) · α = {ALPHA} · "
    f"population mean Edit Index {np.nanmean(EI_PER_SAMPLE):+.3f}",
    fontsize=11.5, y=1.10,
)
fig.tight_layout()
plt.show()

print(f"per-sample Edit Index on these rows: {[f'{EI_PER_SAMPLE[i]:+.2f}' for i in SAMPLES]}   "
      f"population mean {np.nanmean(EI_PER_SAMPLE):+.3f}, median {np.nanmedian(EI_PER_SAMPLE):+.3f}")
print("The two GT curves are identical outside the shaded band — that is what makes the frame look")
print("'nearly right' as a whole. Inside the band the unedited curve (dotted) is the one the model")
print("tracks. The intervention moves the solid coloured curve only slightly toward the target.")

## Summary

### Current results (updated 2026-08-18)

**The probing half of the paper replicates cleanly. The intervention half does not.**

**1 — The probe result replicates (Fig 1, Table 1).** Exactly the paper's §3 finding: the linear
probe is mediocre and the one-hidden-layer MLP is much better — best position R² **0.798 → 0.934**,
a gap of **+0.136**, with the MLP rising monotonically with depth (0.796 at the encoder port to
0.934 at block 3). A world representation is present in the residual stream and it is **nonlinear**,
just as they report.

**2 — The intervention does not (Figs 3, 5, Table 3).** The optimisation succeeds completely: the
probe read-out is driven from **3.35 → 0.007–0.018** sim units, a 99.5% reduction, at every applied
layer. The generation barely responds: Edit Index **−0.684 (unsteered) → −0.538** at the best
applied layer, a gain of **+0.146** on a scale where the two ground-truth worlds sit at ±1. The
waterfall (Fig 3) is unambiguous — the object stays on the **red ghost locator** where it was and
never reaches the **green target**, in every sample and every arm.

**3 — The write is ignored, not destructive, and it reverts within one frame (Fig 6).** Fidelity
ratio is **0.993–0.999** everywhere; no arm comes near the >1.05 degradation guard, so this is not
a case of scoring by wrecking the frame. But the arms **collapse onto the unsteered curve by step 1**
and the gap decays **+0.146 → +0.010** by step 14. In the thread's vocabulary this is *reverts*, not
*collapses* or *drifts*.

**4 — Earlier applied layers propagate further.** Edit Index by starting point: **−0.538** (point 0
and 1) → −0.565 (2) → −0.606 (3) → **−0.622** (point 4). This is the structural prediction: an edit
at residual point ℓ changes block inputs for layers **> ℓ only**, so writing at the last point alters
this position's own prediction and propagates to nothing.

**5 — Reading the entire world state changes nothing (Fig 4, Fig 5b).** The full-state probe
(positions *and* velocities, 8 dims, same edit objective) lands at **−0.539** against the position
probe's −0.538 — a difference of 0.001. Giving the write a complete world state to move does not
help. Worth noting *why* this is a weak test of "completeness": velocity is barely readable at all
(**−0.04 to 0.45** per dim, Table 2), so the extra dims carry little information to begin with.

**6 — The ceiling for any single-frame intervention on this model is itself low.** The oracle
observation — the model simply *shown* the true post-edit frame — reaches only **+0.126**, decaying
to −0.030 by step 14. So the probe write achieves about **18%** of what a perfect single-frame
intervention achieves (+0.146 of +0.810), and even that reference is far from +1.

**7 — Which write you land on depends on the optimiser, and that is informative (Fig 7, Table 4).**
Selecting the step size by read-out error for both, Adam's write is **1.7–4.9× larger in norm** and moves
the generation; plain gradient descent lands the read-out with a smaller write and moves the
generation essentially not at all (at point 0: read-out 0.192, Edit Index **−0.680** = unsteered).
The set of activations satisfying "the probe reads the target" is large, and the probe constraint
does not pin down a member of it that the dynamics honour.

**8 — None of this depends on how much history the model carries (Fig 8).** Gains over each model's
own unsteered index: **+0.153** (window 2) / **+0.137** (window 4) / **+0.146** (window 16), against
best probe R² 0.932/0.931/0.934. Flat.

### Interpretation (not established)

This **does not contradict Li et al.**; it locates where the difference lives. Two candidates, and
this notebook does not separate them:

- **The world.** Othello's board state is a discrete, exactly-determined function of the move
  sequence, and their intervention flips a tile the legal-move computation depends on directly. Here
  the state is continuous and the output is a *rendered* observation, in which moving one object
  requires a coordinated change across many rays. This lines up with the thread's 2026-08-05 result
  that `readable ≠ grabbable` is **inherited from the renderer**, measured with no model involved.
- **The read-out.** Their probe predicts a quantity the next-token computation consumes; ours
  predicts a quantity that is merely *correlated* with what the decoder consumes. §7's optimiser
  result sharpens this: probe-satisfying writes form a large set, most of which the dynamics ignore.

What this does settle for the thread: the failure of probe-derived writes is **not** an artefact of
this repo's editor implementations. The strongest published version of that method, ported with its
schedule, its loss, its multi-layer write, and its own baseline, fails here too.

### Limits, stated plainly

- One dataset (`4_fixed_refl_inview`), one architecture family, **N = 256** edit episodes.
- The probe here is the **paper's** one-hidden-layer MLP, **not** `pim.extractors.fit_readability_probes`
  (2 hidden layers × 256). The R² values are not comparable to readability numbers elsewhere in the repo.
- Held out **by sequence**, unlike the paper's by-frame split. This makes our probe numbers *lower*
  and more honest, but it is a deviation from their protocol.
- The activation update uses Adam by default; §7 quantifies what plain GD does instead rather than
  assuming the paper's robustness claim carries over.